# Real-Time Mode — one line to sub-second streaming

**⏱ ~3 min  ·  🎯 Show that turning on Spark 4.1 Real-Time Mode is a single trigger change  ·  🧰 Setup: none — `Shift`+`Enter`.**

> **Before you start:** run the **warm-up** cell (starts Spark + a stream of order events, ~10s).

**What the audience will see:** order events flowing through guardrails (flag oversized orders, too many items, leaked credentials) and getting routed ALLOW / QUARANTINE — first in micro-batch, then in Real-Time Mode, changing **one line**.

In [ ]:
# Warm-up: Spark + start producing order events to Kafka (~10s)
import sys, time, os
sys.path.insert(0, '/home/jovyan/demos/realtime-mode')
import rtm_stream_lib as rtm
from pyspark.sql import SparkSession
spark = SparkSession.builder.config('spark.sql.streaming.realTimeMode.allowlistCheck','false').getOrCreate()
rtm.start_producer(rows_per_sec=20)
print('producing order events...')

## 1. The guardrails (stateless checks)
Plain column logic — no windows, no state. Stateless is exactly what Real-Time Mode requires.

> 💬 **Say:** "Three checks: oversized total, too many items, a leaked credential. Each event is ALLOW or QUARANTINE."

In [ ]:
from IPython.display import Code
Code(filename='/home/jovyan/demos/realtime-mode/rtm_stream_lib.py')

## 2. The one-line difference
Real-Time Mode isn't a new query — it's the same stream with a different **trigger**:

```python
# Normal micro-batch trigger (what you'd write today):
writer.trigger(processingTime='5 seconds').start()

# Real-Time Mode — the ONLY change:
rt = spark._jvm.org.apache.spark.sql.streaming.Trigger.RealTime('5 seconds')
writer._jwrite.trigger(rt).start()
```
> 💬 **Say:** "There's no PySpark keyword for it yet, so we reach through to the JVM — but it's still one line."

### Baseline: normal micro-batch
> 👀 **Point at:** the ALLOW/QUARANTINE split and a couple of quarantined orders.

In [ ]:
q = rtm.start_query(spark, use_realtime=False, query_name='mb')
time.sleep(12)
spark.sql('SELECT decision, count(*) AS n FROM mb GROUP BY decision').show()
spark.sql("SELECT order_id, brand, total, reasons FROM mb WHERE decision='QUARANTINE'").show(5, truncate=False)
q.stop()

### Flip to Real-Time Mode — change `use_realtime` to `True`
> 💬 **Say:** "Same query, same guardrails. The only thing I changed is the trigger."

In [ ]:
q = rtm.start_query(spark, use_realtime=True, query_name='rt')   # <-- the one change
time.sleep(12)
spark.sql('SELECT decision, count(*) AS n FROM rt GROUP BY decision').show()
spark.sql("SELECT order_id, brand, total, reasons FROM rt WHERE decision='QUARANTINE'").show(5, truncate=False)
q.stop()

### Recap (say this to close)
- The guardrail query **never changed** — only the trigger did.
- `Trigger.RealTime` is Spark 4.1's Real-Time Mode; today it's reached via the JVM (no PySpark keyword yet).
- Real-Time Mode needs a **Kafka source** and **stateless** ops — both true here.